# Báo cáo tổng hợp cuối cùng

Notebook này gom kết quả từ các bước EDA, tiền xử lý, lựa chọn đặc trưng và mô hình hóa để tạo một bản tóm tắt chốt cuối cho dự án Abalone.

## 1. Import thư viện

In [ ]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Xác định thư mục kết quả

In [ ]:
duong_dan_ung_vien = [
    Path('../../outputs/metrics'),
    Path('../outputs/metrics'),
    Path('outputs/metrics'),
    Path('AbaloneAge/outputs/metrics'),
]

thu_muc_metrics = None
for p in duong_dan_ung_vien:
    p_day_du = p.resolve()
    if p_day_du.exists():
        thu_muc_metrics = p_day_du
        break

if thu_muc_metrics is None:
    raise FileNotFoundError(
        'Khong tim thay thu muc outputs/metrics. Hay chay cac notebook truoc truoc khi mo bao cao tong hop.'
    )

thu_muc_hinh = thu_muc_metrics.parent / 'figures'
thu_muc_hinh.mkdir(parents=True, exist_ok=True)

print('Thu muc metrics:', thu_muc_metrics)
print('Thu muc hinh    :', thu_muc_hinh)

## 3. Đọc các summary đã lưu

In [ ]:
tap_tin_json = {
    'outlier': thu_muc_metrics / '01_eda_outlier_summary.json',
    'scaling': thu_muc_metrics / '02_preprocessing_scaling_summary.json',
    'kbest': thu_muc_metrics / '03_kbest_summary.json',
    'baseline': thu_muc_metrics / '03_baseline_summary.json',
    'rfe': thu_muc_metrics / '03_rfe_summary.json',
    'model_based': thu_muc_metrics / '03_model_based_summary.json',
    'compare_v2': thu_muc_metrics / '03_compare_v2_summary.json',
    'ensemble': thu_muc_metrics / '04_model_ensemble_try1_summary.json',
}

tong_hop_json = {}
for ten, p in tap_tin_json.items():
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            tong_hop_json[ten] = json.load(f)
    else:
        tong_hop_json[ten] = None

tong_hop_json

## 4. Đọc bảng metric đã lưu

In [ ]:
tap_tin_csv = {
    'scaling': thu_muc_metrics / '02_preprocessing_scaling_comparison.csv',
    'kbest': thu_muc_metrics / '03_kbest_k_comparison.csv',
    'rfe': thu_muc_metrics / '03_rfe_n_features_comparison.csv',
    'model_based': thu_muc_metrics / '03_model_based_model_metrics.csv',
    'compare_v2': thu_muc_metrics / '03_compare_v2_method_metrics.csv',
    'ensemble': thu_muc_metrics / '04_model_ensemble_try1_comparison.csv',
}

bang_metrics = []
for ten, p in tap_tin_csv.items():
    if p.exists():
        df_tmp = pd.read_csv(p)
        df_tmp['nguon'] = ten
        bang_metrics.append(df_tmp)

if len(bang_metrics) > 0:
    bang_metrics = pd.concat(bang_metrics, ignore_index=True)
else:
    bang_metrics = pd.DataFrame()

bang_metrics

## 5. Tóm tắt dữ liệu và ngoại lệ

In [ ]:
tom_tat_dulieu = []
if tong_hop_json.get('outlier') is not None:
    outlier = tong_hop_json['outlier']
    tom_tat_dulieu.append({
        'hang_muc': 'Tong so dong',
        'gia_tri': outlier.get('tong_so_dong'),
    })
    tom_tat_dulieu.append({
        'hang_muc': 'So dong co outlier',
        'gia_tri': outlier.get('so_dong_co_outlier'),
    })
    tom_tat_dulieu.append({
        'hang_muc': 'Ty le dong co outlier (%)',
        'gia_tri': outlier.get('ty_le_dong_co_outlier_%'),
    })
    tom_tat_dulieu.append({
        'hang_muc': 'Cot nhieu outlier nhat',
        'gia_tri': outlier.get('cot_nhieu_outlier_nhat'),
    })

bang_tom_tat_dulieu = pd.DataFrame(tom_tat_dulieu)
bang_tom_tat_dulieu

## 6. So sánh tiền xử lý và lựa chọn đặc trưng

In [ ]:
bang_tong_hop = []

if tong_hop_json.get('scaling') is not None:
    s = tong_hop_json['scaling']
    bang_tong_hop.append({
        'buoc': 'scaling',
        'ten_tot_nhat': s.get('scaler_tot_nhat'),
        'MAE_test': s.get('test', {}).get('MAE'),
        'RMSE_test': s.get('test', {}).get('RMSE'),
        'R2_test': s.get('test', {}).get('R2'),
    })

if tong_hop_json.get('kbest') is not None:
    s = tong_hop_json['kbest']
    bang_tong_hop.append({
        'buoc': 'kbest',
        'ten_tot_nhat': s.get('k_tot_nhat'),
        'MAE_test': s.get('test', {}).get('MAE'),
        'RMSE_test': s.get('test', {}).get('RMSE'),
        'R2_test': s.get('test', {}).get('R2'),
    })

if tong_hop_json.get('rfe') is not None:
    s = tong_hop_json['rfe']
    bang_tong_hop.append({
        'buoc': 'rfe',
        'ten_tot_nhat': s.get('so_dac_trung_tot_nhat'),
        'MAE_test': s.get('test', {}).get('MAE'),
        'RMSE_test': s.get('test', {}).get('RMSE'),
        'R2_test': s.get('test', {}).get('R2'),
    })

if tong_hop_json.get('model_based') is not None:
    s = tong_hop_json['model_based']
    bang_tong_hop.append({
        'buoc': 'model_based',
        'ten_tot_nhat': s.get('mo_hinh_tot_nhat'),
        'MAE_test': s.get('test', {}).get('MAE'),
        'RMSE_test': s.get('test', {}).get('RMSE'),
        'R2_test': s.get('test', {}).get('R2'),
    })

if tong_hop_json.get('ensemble') is not None:
    s = tong_hop_json['ensemble']
    bang_tong_hop.append({
        'buoc': 'ensemble',
        'ten_tot_nhat': s.get('mo_hinh_tot_nhat'),
        'MAE_test': s.get('test', {}).get('MAE'),
        'RMSE_test': s.get('test', {}).get('RMSE'),
        'R2_test': s.get('test', {}).get('R2'),
    })

bang_tong_hop_mo_hinh = pd.DataFrame(bang_tong_hop)
bang_tong_hop_mo_hinh

## 7. Bảng metric tổng hợp

In [ ]:
if len(bang_metrics) > 0:
    if 'mo_hinh' not in bang_metrics.columns and 'scaler' in bang_metrics.columns:
        bang_metrics['mo_hinh'] = bang_metrics['scaler']

    cot_ten = 'mo_hinh' if 'mo_hinh' in bang_metrics.columns else bang_metrics.columns[0]
    cot_mae = 'MAE' if 'MAE' in bang_metrics.columns else None
    cot_rmse = 'RMSE' if 'RMSE' in bang_metrics.columns else None
    cot_r2 = 'R2' if 'R2' in bang_metrics.columns else None

    cot_hien_thi = [cot_ten] + [c for c in [cot_mae, cot_rmse, cot_r2, 'nguon'] if c is not None and c in bang_metrics.columns]
    bang_metrics_tinh = bang_metrics[cot_hien_thi].copy()
    bang_metrics_tinh = bang_metrics_tinh.sort_values(by=cot_rmse if cot_rmse else cot_ten, ascending=True)
    display(bang_metrics_tinh)
else:
    bang_metrics_tinh = pd.DataFrame()
    print('Khong co file metric nao duoc nap vao.')

## 8. Biểu đồ tổng quan

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if len(bang_tong_hop_mo_hinh) > 0:
    axes[0].barh(bang_tong_hop_mo_hinh['buoc'], bang_tong_hop_mo_hinh['RMSE_test'], color='tomato')
    axes[0].set_title('RMSE test theo nhóm giải pháp')
    axes[0].set_xlabel('RMSE')

    axes[1].barh(bang_tong_hop_mo_hinh['buoc'], bang_tong_hop_mo_hinh['MAE_test'], color='steelblue')
    axes[1].set_title('MAE test theo nhóm giải pháp')
    axes[1].set_xlabel('MAE')

    axes[2].barh(bang_tong_hop_mo_hinh['buoc'], bang_tong_hop_mo_hinh['R2_test'], color='seagreen')
    axes[2].set_title('R2 test theo nhóm giải pháp')
    axes[2].set_xlabel('R2')
else:
    for ax in axes:
        ax.axis('off')

plt.tight_layout()
plt.savefig(thu_muc_hinh / '06_report_summary_metrics.png', dpi=300, bbox_inches='tight')
plt.show()
print('Da luu hinh: 06_report_summary_metrics.png')

## 9. Kết luận ngắn

In [ ]:
if len(bang_tong_hop_mo_hinh) > 0:
    hang_tot_nhat = bang_tong_hop_mo_hinh.sort_values(by='RMSE_test').iloc[0]
    print('Buoc tot nhat theo RMSE test:', hang_tot_nhat['buoc'])
    print('Gia tri tot nhat:', hang_tot_nhat['ten_tot_nhat'])
    print(f"MAE  : {hang_tot_nhat['MAE_test']:.4f}")
    print(f"RMSE : {hang_tot_nhat['RMSE_test']:.4f}")
    print(f"R2   : {hang_tot_nhat['R2_test']:.4f}")
else:
    print('Chua co du lieu tong hop de ket luan.')

## 10. Lưu báo cáo tổng hợp

In [ ]:
thu_muc_bao_cao = Path('../../reports/bao_cao_tieng_viet').resolve()
thu_muc_bao_cao.mkdir(parents=True, exist_ok=True)

if len(bang_tong_hop_mo_hinh) > 0:
    bang_tong_hop_mo_hinh.to_csv(thu_muc_bao_cao / '06_report_summary_model_overview.csv', index=False)

bao_cao = {
    'tong_quan': {
        'co_du_lieu_outlier': tong_hop_json.get('outlier') is not None,
        'co_du_lieu_scaling': tong_hop_json.get('scaling') is not None,
        'co_du_lieu_feature_selection': any(tong_hop_json.get(k) is not None for k in ['baseline', 'kbest', 'rfe', 'model_based', 'compare_v2']),
        'co_du_lieu_ensemble': tong_hop_json.get('ensemble') is not None,
    },
    'tong_hop': bang_tong_hop_mo_hinh.to_dict(orient='records') if len(bang_tong_hop_mo_hinh) > 0 else [],
    'ket_luan': {
        'buoc_tot_nhat': hang_tot_nhat['buoc'] if len(bang_tong_hop_mo_hinh) > 0 else None,
        'gia_tri_tot_nhat': hang_tot_nhat['ten_tot_nhat'] if len(bang_tong_hop_mo_hinh) > 0 else None,
    } if len(bang_tong_hop_mo_hinh) > 0 else {},
}

with open(thu_muc_bao_cao / '06_report_summary.json', 'w', encoding='utf-8') as f:
    json.dump(bao_cao, f, ensure_ascii=False, indent=2)

print('Da luu: 06_report_summary_model_overview.csv')
print('Da luu: 06_report_summary.json')
print('Hoan thanh bao cao tong hop.')